# JavaScript — Dates and JSON

## LESSON 37 — Dates and formatting

```js
const now = new Date();                      // this moment
const launch = new Date("2026-09-10");       // from an ISO string
const exact = new Date(2026, 8, 10);         // year, MONTH INDEX, day
```

Look at that third line: `8` is September. **Months are counted from 0**, and nothing else in the `Date` API is. It is the single most common date bug in JavaScript.

### Reading a date

```js
launch.getFullYear();     // 2026
launch.getMonth();        // 8      <- September, not August
launch.getDate();         // 10     <- day of the month
launch.getDay();          // 4      <- day of the week, 0 = Sunday
launch.getTime();         // milliseconds since 1 January 1970
```

### Comparing and measuring

Dates compare correctly with `<` and `>`, but **not** with `===` — two separate objects are never equal, even for the same instant. Compare `getTime()` values instead.

```js
const days = (b.getTime() - a.getTime()) / (1000 * 60 * 60 * 24);
```

### Intl — showing a date to a human

`toString()` gives you something no user wants to read. `Intl` formats for a place and a language.

```js
new Intl.DateTimeFormat("en-GB").format(launch);       // "10/09/2026"
new Intl.DateTimeFormat("en-US").format(launch);       // "9/10/2026"

new Intl.DateTimeFormat("en-GB", {
  day: "numeric", month: "long", year: "numeric",
}).format(launch);                                     // "10 September 2026"
```

The same object formats numbers and money:

```js
new Intl.NumberFormat("en-GB", { style: "currency", currency: "EUR" })
  .format(1234.5);                                     // "€1,234.50"
```

### Key notes

- **`getMonth()` is zero-based, `getDate()` is not.** January is `0`, but the first of the month is `1`. Expect to be caught by this once.
- `new Date("2026-09-10")` is read as UTC, so in some time zones it prints as the 9th. Pass numbers — `new Date(2026, 8, 10)` — when you mean a local calendar date.
- **Never compare dates with `===`.** Compare `getTime()`, or use `<` and `>`.
- Reach for `Intl` rather than assembling a date string by hand. It knows every locale's ordering, and you do not.

### Example

In [ ]:
const exampleLaunch = new Date(2026, 8, 10);

console.log(exampleLaunch.getFullYear());
console.log(exampleLaunch.getMonth());
console.log(exampleLaunch.getDate());

const exampleEnd = new Date(2026, 8, 24);
const exampleDays = (exampleEnd.getTime() - exampleLaunch.getTime()) / (1000 * 60 * 60 * 24);
console.log(exampleDays);

console.log(new Intl.DateTimeFormat("en-GB").format(exampleLaunch));
console.log(
  new Intl.DateTimeFormat("en-GB", {
    day: "numeric",
    month: "long",
    year: "numeric",
  }).format(exampleLaunch),
);
console.log(
  new Intl.NumberFormat("en-GB", { style: "currency", currency: "EUR" }).format(1234.5),
);

### Exercise

1. Create a date for **1 March 2026**, being careful with the month.
2. Print its year, its month index and its day of the month.
3. Create a second date for **15 March 2026** and print how many days apart they are.
4. Print the first date formatted as `1 March 2026`.
5. Print `2499.9` formatted as euros.

_Don't open `solutions.ipynb` until you've actually tried._

In [7]:
// Your code here
//1
const first= new Date(2026,2,1);
//2
console.log(first.getFullYear(),first.getMonth(),first.getDate());
//3 
const second= new Date(2026,2,15);
console.log((second.getTime()-first.getTime())/(1000*60*60*24));
//4
console.log(
new Intl.DateTimeFormat("en-GB",
        {
            day:"numeric",
            month:"long",
            year:"numeric"
        }
    ).format(first)
);

//5
console.log(
new Intl.NumberFormat("en-GB",
        {
            style:"currency",
            currency:"EUR"
        }
    ).format(1234.43),
);

2026 2 1
14
1 March 2026
€1,234.43


### Mini challenge

Write `daysUntil(target)` that takes a `Date` and returns the whole number of days from **today** until then — negative if the date has passed.

Then write `isWeekend(date)`, returning `true` for Saturday and Sunday. Test it on two dates you know the answer for.

In [17]:
// Your code here
function daysUntil(target){
    const now= new Date();
    const res=(target.getTime()-now.getTime())/(1000*60*60*24);
    return res;
}

console.log(daysUntil(new Date("2026-10-10")));
console.log(daysUntil(new Date("2026-09-15")));


function isWeekend(date){
    /* const days=["sunday","monday","thusday","wendsday","thursday","friday","saturday"];
    return days[date.getDay()];
     */
    return date.getDay()===0 || date.getDay()===6? true:false;
}
console.log(isWeekend(new Date("2026-10-10")));
console.log(isWeekend(new Date("2026-09-15")));



25.161389236111113
0.1613892361111111
true
false


## LESSON 38 — JSON

An object lives in your program's memory. A network sends only text. **JSON** is the agreed way to write an object as text, and it is what every API you will ever call speaks.

```js
const user = { name: "Mia", age: 28, active: true };

const text = JSON.stringify(user);
// '{"name":"Mia","age":28,"active":true}'   <- a string

const back = JSON.parse(text);
// { name: "Mia", age: 28, active: true }    <- an object again
```

| function | direction |
|---|---|
| `JSON.stringify(value)` | object to text — for sending or storing |
| `JSON.parse(text)` | text to object — for reading what arrived |

### Readable output

```js
JSON.stringify(user, null, 2);   // indented by 2 spaces, one key per line
```

### What does not survive the trip

JSON has no way to write a function or a date, so `stringify` drops or changes them, silently.

```js
JSON.stringify({ run: () => {}, when: new Date(), missing: undefined });
// '{"when":"2026-09-10T00:00:00.000Z"}'
```

The function and the `undefined` key vanished. The date became a **string** — parse it back and you get text, not a `Date`.

### Parsing can fail

`JSON.parse` throws on anything that is not valid JSON, which includes the HTML error page a server sends when something has gone wrong.

```js
try {
  JSON.parse(response);
} catch (error) {
  console.log("Not valid JSON:", error.message);
}
```

### Key notes

- **`stringify` silently drops functions and `undefined`.** No warning, no error. If a key disappears on a round trip, this is why.
- **A date does not survive a round trip.** It goes out as a string and comes back as a string. Rebuild it with `new Date(text)`.
- `JSON.parse` **throws**. Any parse of data you did not create belongs in a `try`/`catch` — the next lesson is about exactly that.
- JSON keys must be in double quotes. `{name: "Mia"}` is valid JavaScript and invalid JSON.

### Example

In [18]:
const exampleUser = { name: "Mia", age: 28, active: true };

const exampleText = JSON.stringify(exampleUser);
console.log(exampleText);
console.log(typeof exampleText);

const exampleBack = JSON.parse(exampleText);
console.log(exampleBack.name, typeof exampleBack);

console.log(JSON.stringify(exampleUser, null, 2));

const exampleLossy = JSON.stringify({
  run: () => {},
  when: new Date(2026, 0, 1),
  missing: undefined,
});
console.log(exampleLossy);

try {
  JSON.parse("<html>not json</html>");
} catch (error) {
  console.log("Not valid JSON:", error.message);
}

{"name":"Mia","age":28,"active":true}
string
Mia object
{
  "name": "Mia",
  "age": 28,
  "active": true
}
{"when":"2025-12-31T23:00:00.000Z"}
Not valid JSON: Unexpected token '<', "<html>not "... is not valid JSON


### Exercise

Given:

```js
const order = { id: 7, items: ["pen", "book"], total: 24.5 };
```

1. Print it as a JSON string.
2. Print it again, indented by 2 spaces.
3. Parse your string back into an object and print the first item.
4. Print `typeof` the string and `typeof` the parsed object, and compare them.
5. Stringify `{ save: () => {}, note: undefined, id: 1 }` and say in a comment what survived.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here
const order = { id: 7, items: ["pen", "book"], total: 24.5 };
//1
const orderText=JSON.stringify(order);
console.log(orderText);
//2
console.log(JSON.stringify(order,null,2));
//3
const parseOrder=JSON.parse(orderText);
console.log(parseOrder);
//4
console.log(typeof orderText, typeof parseOrder);
//5
const brokeText=JSON.stringify({ save: () => {}, note: undefined, id: 1 });
console.log(brokeText); // only id:1 survived


{"id":7,"items":["pen","book"],"total":24.5}
{
  "id": 7,
  "items": [
    "pen",
    "book"
  ],
  "total": 24.5
}
{ id: 7, items: [ "pen", "book" ], total: 24.5 }
string object
{"id":1}


### Mini challenge

1. Write `safeParse(text)` that returns the parsed object, or `null` when the text is not valid JSON — without ever crashing.
2. Call it once with valid JSON and once with `"oops"`, printing both results.
3. Stringify an object containing a `Date`, parse it back, and print `typeof` the date field. Then convert it back into a real `Date` and print its year.

In [50]:
// Your code here
//1
function safeParse(test){
    try{
        return JSON.parse(test);
    }catch{
        return null;
    }
}
//2
const res=safeParse(`{"id":"1"}`);
console.log(res,typeof res);
const res2=safeParse("opps");
console.log(res2,typeof res2);
//3
const res3={
    date:new Date()
}
console.log(res3);
const res3Text=JSON.stringify(res3);
console.log(res3Text);
const res3Parse=safeParse(res3Text);
console.log(res3Parse, typeof res3Parse.date);
const realDate=new Date(res3Parse.date);
console.log(realDate.getFullYear());

{ id: "1" } object
null object
{ date: 2026-09-14T20:35:02.740Z }
{"date":"2026-09-14T20:35:02.740Z"}
{ date: "2026-09-14T20:35:02.740Z" } string
2026
